# GUARACI em 5 minutos

**GUARACI** é uma plataforma Python de quimiometria multitécnica, aberta e reprodutível, para classificação, autenticação, exploração e quantificação de matrizes complexas (FT-NIR, NIR, MIR, Raman, HPLC, GC-MS, RMN...).

O diferencial: **validação group-aware** (`GroupKFold`/LOGO por `mae_id`) impede que réplicas físicas da mesma amostra apareçam em treino e teste ao mesmo tempo — um vazamento comum e sub-relatado na literatura de espectroscopia, que infla acurácia artificialmente.

Este notebook roda o pipeline completo **sem nenhum dado seu** — usa um gerador de espectros sintéticos (com réplicas físicas simuladas) já testado no repositório.

Repositório: https://github.com/ErleySC/guaraci

> Este notebook instala a partir do GitHub (`pip install git+...`) enquanto o pacote não é publicado no PyPI. Assim que a publicação acontecer, troque a célula de instalação por `pip install guaraci-chemometrics`.

In [ ]:
# Instalação (GitHub por enquanto; troque por 'pip install guaraci-chemometrics' após a publicação no PyPI)
!pip install -q "git+https://github.com/ErleySC/guaraci.git"

In [ ]:
import guaraci.pipeline as pq

print(f"GUARACI v{pq.__version__}")

## 1. Diagnóstico do ambiente

`guaraci doctor` checa Python, RAM/CPU/disco e dependências obrigatórias/opcionais. Útil antes de abrir um issue de instalação.

In [ ]:
!guaraci doctor

## 2. Rodando o pipeline com dados sintéticos

`modo="sintetico"` gera espectros de 3 espécies fictícias, cada uma com réplicas físicas (`mae_id`) e amostras puras/adulteradas — o suficiente para exercitar classificação (PLS-DA) **e** o diferencial do projeto: DD-SIMCA com sensibilidade estimada por leave-one-group-out (LOGO), não por re-substituição.

Isso roda em ~20-30 segundos.

In [ ]:
from pathlib import Path
import os

saida = Path.cwd() / "GUARACI_Demo"

cfg = pq.Config(
    pasta_entrada=str(saida / "dados_dummy"),  # modo sintetico ignora isto
    pasta_saida_raiz=str(saida),
    modo="sintetico",
    tag="colab_demo",
    nivel="N2",               # DD-SIMCA + sensibilidade LOGO (o diferencial central)
    objetivo="auto",
    n_por_classe=15,
    n_pontos_sint=300,
    wn_min=400.0,
    wn_max=4001.0,
    sint_adulterantes=("S", "M", "A"),
    max_lvs=15,
    n_splits_cv=3,
    n_repeats_cv=1,
    n_permutacoes=50,
    n_permutacoes_wold=50,
    n_bootstrap_vip=10,
    n_bootstrap_bca=100,
    n_monte_carlo=20,
    executar_benchmark=False,
    executar_monte_carlo=False,
    executar_shap=False,
    executar_wold=False,
    executar_cv_anova=False,
    executar_opls=False,
    executar_etapa4=False,
    comparar_pipelines=False,
    comparar_hca_pipelines=False,
)
os.makedirs(cfg.pasta_entrada, exist_ok=True)

pq.executar(cfg)

In [ ]:
# Encontra a pasta da execução (PLSDA_OE_...) gerada acima
runs = []
for raiz, dirs, _arqs in os.walk(str(saida)):
    for d in dirs:
        if d.startswith("PLSDA_OE_"):
            runs.append(os.path.join(raiz, d))
pasta_run = Path(sorted(runs)[-1])
print("Resultados em:", pasta_run)

## 3. O diferencial: sensibilidade DD-SIMCA por LOGO

Repare no aviso: com poucos grupos de réplica por espécie, a sensibilidade é **explicitamente marcada como incerta**, em vez de reportar um número de re-substituição inflado (~100%) sem avisar ninguém. Ver `docs/VALIDATION.md` no repositório para a validação desse método contra a referência (Pomerantsev & Rodionova, 2014).

In [ ]:
resumo = pasta_run / pq.NOME_RELATORIOS / "resumo_modelo.txt"
print(resumo.read_text(encoding="utf-8")[:4000])

## 4. Visualizando as figuras

Todas as figuras ficam em `Graficos/` dentro da pasta da execução.

In [ ]:
from IPython.display import Image, display

pasta_figs = pasta_run / pq.NOME_GRAFICOS
for nome in ["fig0_espectros_medios_classe.png", "fig1_pca_scores.png",
             "fig2_plsda_scores.png", "fig_sprint3_ddsimca_acceptance.png"]:
    caminho = pasta_figs / nome
    if caminho.exists():
        print(nome)
        display(Image(filename=str(caminho)))

## Próximos passos

- Rodar com dados reais: troque `modo="sintetico"` por `modo="dx"` (JCAMP-DX) ou `modo="csv"` e aponte `pasta_entrada` para os seus espectros.
- Interface interativa (Rich CLI): `guaraci` no terminal, sem argumentos.
- Interface web (Streamlit): `pip install guaraci-chemometrics[web]` e `streamlit run app_quimiometria.py`.
- Validação contra implementações de referência: `docs/VALIDATION.md`.
- Segurança ao carregar modelos `.joblib` de terceiros: `docs/SECURITY.md`.
- Limitações conhecidas (o que este software **não** faz bem ainda): seção 9 de `docs/MANUAL.md`.

Citação: ver `README.md` (APA/ABNT/BibTeX) ou `paper/paper.md` (submissão JOSS).